# 3D Point Cloud + Gaussian Splatting from RE10K Image & Trajectory

**Inputs**
- `project_data/bedroom.jpg` — starting frame (1878×1050, used as high-quality anchor).
- `project_data/fff9864727c42c80.txt` — RealEstate10K camera trajectory.
- `project_data/output64.mp4` — 64-frame video along that trajectory (DFoT output).

**Pipeline**
1. Parse RE10K trajectory → per-frame intrinsics + extrinsics.
2. Extract video frames and align to pose count.
3. Build a Nerfstudio `transforms.json` dataset.
4. **[NEW]** Run **Depth Anything V2 Metric Indoor** on every frame + full-res bedroom.jpg.
5. **[NEW]** Align depth scale to the RE10K camera trajectory metric.
6. **[NEW]** Unproject depth maps → dense coloured 3D point cloud (Open3D).
7. **[NEW]** Convert point cloud → 3DGS-format init PLY (positions, SH colour, scale, rotation).
8. Train 3D Gaussian Splatting with [Brush](https://github.com/ArthurBrussee/brush) (Metal-native), using the depth init.

> **Why Depth Anything V2?**  
> Without depth priors Brush initialises Gaussians randomly → noise.  
> Unprojecting per-frame metric depth gives geometrically correct initialisation.

In [9]:
import os, sys, json, math, shutil, subprocess, urllib.request, tarfile
from pathlib import Path
import numpy as np

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs],
                   check=True, capture_output=True)

for _pkg, _mod in [("transformers>=4.40 accelerate", "transformers"),
                   ("open3d", "open3d"), ("tqdm", "tqdm"), ("plyfile", "plyfile")]:
    try:
        __import__(_mod.split(".")[0])
    except ImportError:
        print(f"Installing {_mod}…"); _pip(*_pkg.split())

BASE_DIR     = Path(os.getcwd()).resolve()
PROJECT_DATA = BASE_DIR / "project_data"
TOOLS_DIR    = BASE_DIR / "tools"
WORK_DIR     = BASE_DIR / "workdir_gs"
DATASET_DIR  = WORK_DIR / "dataset"
FRAMES_DIR   = DATASET_DIR / "images"
EXPORT_DIR   = WORK_DIR / "splats"
for p in (WORK_DIR, DATASET_DIR, FRAMES_DIR, EXPORT_DIR, TOOLS_DIR):
    p.mkdir(parents=True, exist_ok=True)

START_IMAGE  = PROJECT_DATA / "bedroom.jpg"
TRAJECTORY   = PROJECT_DATA / "fff9864727c42c80.txt"
DFOT_VIDEO   = PROJECT_DATA / "dfot_bedroom.mp4"
FALLBACK_VID = PROJECT_DATA / "output64.mp4"
VIDEO_PATH   = DFOT_VIDEO if DFOT_VIDEO.exists() else FALLBACK_VID

assert START_IMAGE.exists(), f"missing {START_IMAGE}"
assert TRAJECTORY.exists(),  f"missing {TRAJECTORY}"
assert VIDEO_PATH.exists(),  f"missing {VIDEO_PATH}"

print("Start image :", START_IMAGE)
print("Trajectory  :", TRAJECTORY)
print("Video       :", VIDEO_PATH)

Start image : /Users/nidhimathihalli/Downloads/drive-download-20260507T233133Z-3-001/project_data/bedroom.jpg
Trajectory  : /Users/nidhimathihalli/Downloads/drive-download-20260507T233133Z-3-001/project_data/fff9864727c42c80.txt
Video       : /Users/nidhimathihalli/Downloads/drive-download-20260507T233133Z-3-001/project_data/output64.mp4


## 1. Parse RE10K trajectory

Each non-URL line: `timestamp  fx fy cx cy  0 0  r11 r12 r13 t1  r21 r22 r23 t2  r31 r32 r33 t3`  
Intrinsics are **normalised**; the 3×4 matrix is **world→camera** in **OpenCV** convention.

In [10]:
def parse_re10k(path):
    intr, w2c, ts = [], [], []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("http"):
            continue
        parts = line.split()
        if len(parts) != 19:
            continue
        nums = list(map(float, parts))
        ts.append(int(nums[0]))
        intr.append(nums[1:5])
        rt = np.array(nums[7:19], dtype=np.float64).reshape(3, 4)
        m  = np.eye(4); m[:3, :4] = rt
        w2c.append(m)
    return (np.asarray(intr, dtype=np.float64),
            np.asarray(w2c,  dtype=np.float64),
            np.asarray(ts,   dtype=np.int64))

intr_n, w2c_all, ts = parse_re10k(TRAJECTORY)
print(f"Trajectory: {len(w2c_all)} poses")
print("mean intrinsics (fx_n, fy_n, cx_n, cy_n):", intr_n.mean(0).round(4).tolist())

Trajectory: 109 poses
mean intrinsics (fx_n, fy_n, cx_n, cy_n): [0.4862, 0.8644, 0.5, 0.5]


## 2. Extract video frames

In [11]:
import cv2

for f in FRAMES_DIR.glob("*"):
    f.unlink()

cap = cv2.VideoCapture(str(VIDEO_PATH))
all_frames_bgr = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    all_frames_bgr.append(frame)
cap.release()

H, W = all_frames_bgr[0].shape[:2]
print(f"Loaded {len(all_frames_bgr)} frames at {W}×{H}")

n_take    = min(len(all_frames_bgr), len(w2c_all))
frame_idx = np.linspace(0, len(all_frames_bgr) - 1, n_take).round().astype(int)
pose_idx  = np.linspace(0, len(w2c_all) - 1, n_take).round().astype(int)

saved = []
for i, (fi, pi) in enumerate(zip(frame_idx, pose_idx)):
    name = f"frame_{i:04d}.png"
    cv2.imwrite(str(FRAMES_DIR / name), all_frames_bgr[fi])
    saved.append((name, pi))

selected_w2c    = w2c_all[[pi for _, pi in saved]]
selected_intr_n = intr_n[[pi for _, pi in saved]]
print(f"Wrote {len(saved)} frames  →  {FRAMES_DIR}")

Loaded 64 frames at 256×256
Wrote 64 frames  →  /Users/nidhimathihalli/Downloads/drive-download-20260507T233133Z-3-001/workdir_gs/dataset/images


## 3. Build Nerfstudio `transforms.json`

Invert world→cam, flip y/z (OpenCV→OpenGL convention for Nerfstudio/Brush).

In [12]:
OPENCV_TO_OPENGL = np.diag([1.0, -1.0, -1.0, 1.0])

fx_n, fy_n, cx_n, cy_n = selected_intr_n.mean(axis=0)
fl_x = fx_n * W
fl_y = fy_n * H
cx   = cx_n * W
cy   = cy_n * H

frames_meta = []
for (name, _), w2c in zip(saved, selected_w2c):
    c2w_cv = np.linalg.inv(w2c)
    c2w_gl = c2w_cv @ OPENCV_TO_OPENGL
    frames_meta.append({
        "file_path": f"images/{name}",
        "transform_matrix": c2w_gl.tolist(),
    })

transforms = {
    "camera_model": "OPENCV",
    "fl_x": fl_x, "fl_y": fl_y, "cx": cx, "cy": cy,
    "w": W, "h": H,
    "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
    "frames": frames_meta,
}
(DATASET_DIR / "transforms.json").write_text(json.dumps(transforms, indent=2))
print(f"transforms.json  fl_x={fl_x:.1f}  fl_y={fl_y:.1f}  {W}×{H}  {len(frames_meta)} frames")

transforms.json  fl_x=124.5  fl_y=221.3  256×256  64 frames


## 4. Depth estimation — Depth Anything V2

**Depth Anything V2 Metric Indoor** (preferred): outputs depth in metres, consistent across frames.  
Falls back to the relative-depth Large model + trajectory-based scale estimation.

We run on:
- All 64 video frames (256×256)
- `bedroom.jpg` at its native 1878×1050 resolution (used as the high-quality anchor for pose 0)

In [13]:
import torch
from PIL import Image as PILImage
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

device = (
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {device}")

METRIC_MODEL   = "depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf"
RELATIVE_MODEL = "depth-anything/Depth-Anything-V2-Large-hf"

def _load_depth_model(name):
    proc = AutoImageProcessor.from_pretrained(name)
    mdl  = AutoModelForDepthEstimation.from_pretrained(
               name, dtype=torch.float32).to(device).eval()
    return proc, mdl

try:
    depth_proc, depth_mdl = _load_depth_model(METRIC_MODEL)
    is_metric = True
    print("Loaded:", METRIC_MODEL)
except Exception as e:
    print(f"Metric model unavailable ({e})\nFalling back to relative model…")
    depth_proc, depth_mdl = _load_depth_model(RELATIVE_MODEL)
    is_metric = False
    print("Loaded:", RELATIVE_MODEL)

Device: mps


Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

Loaded: depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf


In [14]:
from tqdm.auto import tqdm

def _infer_depth(pil_img, out_hw):
    """Run depth model on a PIL image; return float32 numpy array resized to out_hw=(H,W)."""
    inputs = depth_proc(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        pred = depth_mdl(**inputs).predicted_depth
    d = pred.squeeze().float().cpu().numpy()
    d = cv2.resize(d, (out_hw[1], out_hw[0]), interpolation=cv2.INTER_LINEAR)
    return np.clip(d, 0.0, None).astype(np.float32)

# ── Video frames (256×256) ────────────────────────────────────────────────────
frame_paths    = sorted(FRAMES_DIR.glob("frame_*.png"))
depth_maps_raw = []
for fp in tqdm(frame_paths, desc="Depth Anything V2 (video frames)"):
    depth_maps_raw.append(_infer_depth(PILImage.open(fp).convert("RGB"), (H, W)))
depth_maps_raw = np.array(depth_maps_raw, dtype=np.float32)
print(f"Video depth maps : {depth_maps_raw.shape}  "
      f"range [{depth_maps_raw.min():.3f}, {depth_maps_raw.max():.3f}]")

# ── High-res bedroom.jpg at its native resolution ─────────────────────────────
print("Running on full-resolution bedroom.jpg…")
hr_pil   = PILImage.open(START_IMAGE).convert("RGB")
HR_W, HR_H = hr_pil.size          # (1878, 1050)
depth_hr = _infer_depth(hr_pil, (HR_H, HR_W))   # shape (HR_H, HR_W)
print(f"HR depth : {depth_hr.shape}  range [{depth_hr.min():.3f}, {depth_hr.max():.3f}]")

Depth Anything V2 (video frames):   0%|          | 0/64 [00:00<?, ?it/s]

Video depth maps : (64, 256, 256)  range [0.300, 6.068]
Running on full-resolution bedroom.jpg…
HR depth : (1050, 1878)  range [0.732, 5.774]


## 5. Depth scale alignment

RE10K camera positions are in approximately metres indoors.  
The metric model should match; we verify the ratio and correct if mismatched.

In [15]:
def cam_centers(w2c_mats):
    return np.array([-m[:3, :3].T @ m[:3, 3] for m in w2c_mats])

cam_c      = cam_centers(selected_w2c)
cam_extent = cam_c.max(axis=0) - cam_c.min(axis=0)
cam_scale  = float(np.linalg.norm(cam_extent))
print(f"Camera trajectory extent : {cam_extent.round(3)}")
print(f"Trajectory total range   : {cam_scale:.3f} units")

global_median = float(np.median(depth_maps_raw[depth_maps_raw > 0.05]))
print(f"Median raw depth         : {global_median:.3f}")

# RE10K trajectory scale is arbitrary SfM scale (not metric), so we cannot
# compare it to metric depth. When the model is metric, trust it directly.
if is_metric:
    depth_scale = 1.0
    print(f"Metric depth — using raw model output (scale=1.0).")
else:
    # Relative model: align depth median to a plausible indoor range.
    expected_depth = cam_scale * 3.0
    depth_scale = expected_depth / max(global_median, 1e-6)
    print(f"Relative depth: applying scale factor {depth_scale:.4f}")

depth_maps = np.clip(depth_maps_raw * depth_scale, 0.05, 25.0)
depth_hr   = np.clip(depth_hr       * depth_scale, 0.05, 25.0)
D_MIN = 0.05

print(f"Scaled depth range : [{depth_maps.min():.3f}, {depth_maps.max():.3f}]  "
      f"median={np.median(depth_maps):.3f}")

AttributeError: 'numpy.ndarray' object has no attribute 'ptp'

## 6. Build the dense 3D point cloud

**Key insight**: DFoT video frames are synthetic — depth estimates from them disagree with the RE10K poses in 3D space, producing scattered noise instead of geometry.

`bedroom.jpg` is the only real photograph with accurate depth. We use **only that frame** for the point cloud (1878×1050, every pixel unprojected).

> The video frames + their depths are still used by Brush for GS training.

In [ ]:
# Only bedroom.jpg is used for geometry — real photo → trustworthy depth.
# D_MAX=6 m clips far-away unreliable depths (max realistic bedroom depth).
D_MIN, D_MAX = 0.15, 6.0

# Use pose 0 (first trajectory pose) which corresponds to the starting image.
c2w_hr  = np.linalg.inv(selected_w2c[0])
fl_x_hr = fx_n * HR_W;  fl_y_hr = fy_n * HR_H
cx_hr   = cx_n * HR_W;  cy_hr   = cy_n * HR_H
img_hr  = np.array(hr_pil)   # RGB uint8 (HR_H, HR_W, 3)

# Clip depth to bedroom range
depth_hr_clipped = np.clip(depth_hr, D_MIN, D_MAX)

# Unproject every pixel
ys, xs  = np.mgrid[0:HR_H, 0:HR_W]
d_flat  = depth_hr_clipped.ravel()
ok      = (d_flat > D_MIN) & (d_flat < D_MAX)
x_v     = xs.ravel()[ok].astype(np.float64)
y_v     = ys.ravel()[ok].astype(np.float64)
d_v     = d_flat[ok].astype(np.float64)

xc    = (x_v - cx_hr)  / fl_x_hr * d_v
yc    = (y_v - cy_hr)  / fl_y_hr * d_v
pts_h = np.stack([xc, yc, d_v, np.ones_like(d_v)], axis=1)
pts_w = (c2w_hr @ pts_h.T).T[:, :3]
cols  = img_hr.reshape(-1, 3)[ok].astype(np.float64) / 255.0

print(f'Raw points from bedroom.jpg: {len(pts_w):,}')


In [ ]:
import open3d as o3d

VOX_SZ = 0.01   # 1 cm — also used in the GS init step below

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(pts_w)
pcd.colors = o3d.utility.Vector3dVector(np.clip(cols, 0, 1))

pcd = pcd.voxel_down_sample(voxel_size=VOX_SZ)
print(f'After 1 cm voxel downsample: {len(pcd.points):,}')

pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=1.5)
print(f'After outlier removal       : {len(pcd.points):,}')

cam_c0 = -selected_w2c[0][:3,:3].T @ selected_w2c[0][:3,3]
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
pcd.orient_normals_towards_camera_location(cam_c0)

pcd_path = EXPORT_DIR / 'bedroom_depth_cloud.ply'
o3d.io.write_point_cloud(str(pcd_path), pcd, write_ascii=False)
sz_mb = pcd_path.stat().st_size / 1e6
print(f'\n✓ Dense point cloud → {pcd_path.name}  ({sz_mb:.2f} MB, {len(pcd.points):,} pts)')
print(f'  open3d : python /tmp/view_ply.py "{pcd_path}"')
print(f'  MeshLab: open -a MeshLab "{pcd_path}"')


### Quick preview: depth maps + camera trajectory

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(18, 4))
vmax_d = float(np.percentile(depth_maps, 95))

for col, idx in enumerate(np.linspace(0, len(depth_maps) - 1, 4, dtype=int)):
    ax = fig.add_subplot(1, 6, col + 1)
    ax.imshow(depth_maps[idx], cmap="plasma", vmin=D_MIN, vmax=vmax_d)
    ax.set_title(f"Frame {idx} depth", fontsize=9)
    ax.axis("off")

ax3 = fig.add_subplot(1, 6, 5, projection="3d")
ax3.plot(*cam_c.T, "b.-", markersize=2, linewidth=0.8)
ax3.scatter(*cam_c[0],  c="g", s=40, zorder=5, label="start")
ax3.scatter(*cam_c[-1], c="r", s=40, zorder=5, label="end")
ax3.set_title("Camera path", fontsize=9)
ax3.legend(fontsize=7)

ax_hr = fig.add_subplot(1, 6, 6)
hr_thumb = cv2.resize(depth_hr, (256, int(256 * HR_H / HR_W)))
ax_hr.imshow(hr_thumb, cmap="plasma", vmin=D_MIN, vmax=vmax_d)
ax_hr.set_title("bedroom.jpg depth", fontsize=9)
ax_hr.axis("off")

plt.tight_layout()
plt.savefig(str(EXPORT_DIR / "depth_preview.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Convert point cloud → 3DGS initialisation PLY

Each point becomes a tiny isotropic Gaussian (low opacity, unit quaternion rotation).  
Brush will refine all attributes during training.

In [ ]:
from plyfile import PlyData, PlyElement

pts_np  = np.asarray(pcd.points,  dtype=np.float32)
cols_np = np.asarray(pcd.colors,  dtype=np.float32)
nrm_np  = np.asarray(pcd.normals, dtype=np.float32)
N = len(pts_np)

SH_C0    = 0.28209479177387814
f_dc     = (cols_np - 0.5) / SH_C0
f_rest   = np.zeros((N, 45), dtype=np.float32)
opacity  = np.full(N, np.log(0.1 / 0.9), dtype=np.float32)   # logit(0.1)
log_s    = float(np.log(max(VOX_SZ * 0.3, 1e-6)))
scale    = np.full((N, 3), log_s, dtype=np.float32)
rot      = np.zeros((N, 4), dtype=np.float32)
rot[:, 0] = 1.0

dt = ([( "x","f4"),("y","f4"),("z","f4"),("nx","f4"),("ny","f4"),("nz","f4")]
      + [(f"f_dc_{i}",   "f4") for i in range(3)]
      + [(f"f_rest_{i}", "f4") for i in range(45)]
      + [("opacity","f4")]
      + [(f"scale_{i}",  "f4") for i in range(3)]
      + [(f"rot_{i}",    "f4") for i in range(4)])

arr = np.empty(N, dtype=dt)
arr["x"],arr["y"],arr["z"]   = pts_np.T
arr["nx"],arr["ny"],arr["nz"] = nrm_np.T
for i in range(3):  arr[f"f_dc_{i}"]   = f_dc[:, i]
for i in range(45): arr[f"f_rest_{i}"] = f_rest[:, i]
arr["opacity"] = opacity
for i in range(3):  arr[f"scale_{i}"] = scale[:, i]
for i in range(4):  arr[f"rot_{i}"]   = rot[:, i]

gs_init_path = EXPORT_DIR / "bedroom_gs_init.ply"
PlyData([PlyElement.describe(arr, "vertex")]).write(str(gs_init_path))
print(f"✓ 3DGS init PLY  →  {gs_init_path.name}  "
      f"({gs_init_path.stat().st_size/1e6:.2f} MB, {N:,} Gaussians)")

## 8. Get the Brush binary (Metal-native 3DGS for macOS)

In [ ]:
BRUSH_VERSION = "v0.3.0"
BRUSH_BIN = TOOLS_DIR / "brush-app-aarch64-apple-darwin" / "brush_app"

def ensure_brush():
    if BRUSH_BIN.exists():
        return BRUSH_BIN
    url = (f"https://github.com/ArthurBrussee/brush/releases/download/"
           f"{BRUSH_VERSION}/brush-app-aarch64-apple-darwin.tar.xz")
    archive = TOOLS_DIR / "brush.tar.xz"
    print("Downloading", url)
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive, "r:xz") as tf:
        tf.extractall(TOOLS_DIR)
    archive.unlink()
    BRUSH_BIN.chmod(0o755)
    subprocess.run(["xattr", "-d", "com.apple.quarantine", str(BRUSH_BIN)],
                   check=False, stderr=subprocess.DEVNULL)
    return BRUSH_BIN

brush = ensure_brush()
print("brush_app:", brush)
result = subprocess.run([str(brush), "--help"], capture_output=True, text=True)
help_text = result.stdout + result.stderr
print(help_text[:2000])

## 9. Train Gaussian Splatting

Brush is run headless. The 3DGS init PLY is passed via `--init-ply` if supported.

In [ ]:
TOTAL_STEPS  = 15_000
MAX_SPLATS   = 500_000
EXPORT_EVERY = TOTAL_STEPS

# Only delete numbered checkpoint PLYs, not the init PLY
for old in EXPORT_DIR.glob("bedroom_[0-9]*.ply"):
    old.unlink()

supports_init_ply = "--init-ply" in help_text or "init_ply" in help_text
print("Brush supports --init-ply:", supports_init_ply)

cmd = [
    str(brush), str(DATASET_DIR),
    "--total-steps",  str(TOTAL_STEPS),
    "--max-splats",   str(MAX_SPLATS),
    "--export-every", str(EXPORT_EVERY),
    "--export-path",  str(EXPORT_DIR),
    "--export-name",  "bedroom_{iter}.ply",
    "--eval-every",   "1000000",
]
if supports_init_ply:
    cmd += ["--init-ply", str(gs_init_path)]
    print("Using depth-based 3DGS init PLY.")
else:
    print("Brush does not expose --init-ply; depth point cloud is the primary output.")

print("\n$", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
finally:
    proc.wait()
print("\nBrush exit code:", proc.returncode)

## 10. Results

In [ ]:
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)
for p in sorted(EXPORT_DIR.glob("*.ply"), key=lambda x: x.stat().st_size, reverse=True):
    tag = ("  ← dense depth point cloud  [PRIMARY]"
           if "depth_cloud" in p.name else
           "  ← 3DGS init PLY" if "gs_init" in p.name else
           "  ← Brush Gaussian Splat")
    print(f"  {p.name:42s} {p.stat().st_size/1e6:6.2f} MB{tag}")

pcd_out = EXPORT_DIR / "bedroom_depth_cloud.ply"
if pcd_out.exists():
    print()
    print("── View depth point cloud ──")
    print(f'  python -c "import open3d as o3d; '
          f'o3d.visualization.draw_geometries([o3d.io.read_point_cloud(\'\'{pcd_out}\'\')])"')
    print(f'  open -a MeshLab "{pcd_out}"')

gs_plys = sorted(EXPORT_DIR.glob("bedroom_[0-9]*.ply"), key=lambda p: p.stat().st_mtime)
if gs_plys:
    print()
    print("── View Gaussian Splat in Brush viewer ──")
    print(f'  {brush} --with-viewer "{gs_plys[-1]}"')